# SEC Filings Downloader and Cleaner

This notebook demonstrates how to use the `SECFilingsProcessor` class to download and clean SEC filings.

## Features

- **Downloads ALL filings from SEC quarterly index files** (complete population)
- Downloads SEC filings for specified form types and years
- Organizes files into `{year}/Raw/` and `{year}/Clean/` folders
- Removes all HTML, XBRL, and XML tags from filings
- Supports filtering by CIK (company identifier)

## Installation

First, install the required dependencies:

In [ ]:
# Install required packages (uncomment to run)
# !pip install requests beautifulsoup4 lxml

## Import the Processor

In [ ]:
from sec_filings_downloader import SECFilingsProcessor

## Example 1: Download ALL Filings from SEC Index Files (RECOMMENDED)

**This is the recommended method for downloading the complete population of filings.**

The `process_all_filings_from_index` method downloads SEC quarterly index files and extracts ALL filings matching your criteria. This gives you access to the entire EDGAR database for a given year and form type(s).

### Download All 10-K Filings

In [ ]:
# Initialize the processor
processor = SECFilingsProcessor()

# Download ALL 10-K filings from 2023
# Remove max_filings parameter to download the complete population
processor.process_all_filings_from_index(
    form_types=['10-K'],
    year=2023,
    max_filings=50  # Optional: remove this to download ALL filings
)

### Download Multiple Form Types from Index

In [ ]:
processor = SECFilingsProcessor()

# Download ALL 10-K and 10-Q filings
processor.process_all_filings_from_index(
    form_types=['10-K', '10-Q'],
    year=2023,
    max_filings=100  # Optional limit for testing
)

### Download Filings from Specific Quarters Only

In [ ]:
processor = SECFilingsProcessor()

# Download only Q1 and Q2 filings
processor.process_all_filings_from_index(
    form_types=['10-K'],
    year=2023,
    quarters=[1, 2],  # Only Q1 and Q2
    max_filings=50
)

### Download the Complete Population (No Limit)

**Warning:** This will download ALL filings of the specified type(s) from the entire year. This could be thousands of filings and take several hours.

In [ ]:
processor = SECFilingsProcessor()

# Download ALL 10-K filings from 2023 (no limit)
processor.process_all_filings_from_index(
    form_types=['10-K'],
    year=2023
    # No max_filings parameter = download everything!
)

## Example 2: Download Filings for Specific Companies

If you only need filings from specific companies, use the `process_filings` method with a CIK list.

### Single Company

In [ ]:
# Initialize the processor
processor = SECFilingsProcessor()

# Download and clean filings for Apple
processor.process_filings(
    form_types=['10-K'],      # Form types to download
    year=2023,                # Year of filings
    cik_list=['0000320193']   # Apple's CIK
)

### Multiple Companies

In [ ]:
processor = SECFilingsProcessor()

# Download filings for Apple and Microsoft
processor.process_filings(
    form_types=['10-K', '10-Q'],
    year=2023,
    cik_list=[
        '0000320193',  # Apple Inc.
        '0000789019'   # Microsoft Corporation
    ]
)

## Example 3: Download 8-K Current Reports

Download 8-K current report filings:

In [ ]:
processor = SECFilingsProcessor()

# Download 8-K filings using index method
processor.process_all_filings_from_index(
    form_types=['8-K'],
    year=2023,
    max_filings=100
)

## Example 4: Specify a Custom Output Directory

In [ ]:
# Create processor with custom base directory
processor = SECFilingsProcessor(base_dir="./my_sec_data")

processor.process_all_filings_from_index(
    form_types=['10-K'],
    year=2023,
    max_filings=20
)

# This will create:
# ./my_sec_data/2023/Raw/
# ./my_sec_data/2023/Clean/

## Understanding the Output

After running the processor, you'll have:

1. **Raw folder** (`{year}/Raw/`): Contains the original HTML/XBRL filings as downloaded from SEC EDGAR
2. **Clean folder** (`{year}/Clean/`): Contains text-only versions with all HTML/XML/XBRL tags removed

File naming convention: `{CIK}_{FormType}_{FilingDate}.html` (raw) or `.txt` (clean)

## Common CIK Numbers

Here are some commonly used CIK numbers:

- Apple Inc.: 0000320193
- Microsoft Corporation: 0000789019
- Amazon.com Inc.: 0001018724
- Tesla Inc.: 0001318605
- Alphabet Inc.: 0001652044
- Meta Platforms Inc.: 0001326801
- NVIDIA Corporation: 0001045810
- Berkshire Hathaway: 0001067983

You can find CIK numbers by searching on the SEC EDGAR website: https://www.sec.gov/edgar/searchedgar/companysearch.html

## Reading a Cleaned Filing

In [ ]:
# Example: Read and display the first 1000 characters of a cleaned filing
import os

# Get the first cleaned file
clean_folder = "2023/Clean"
if os.path.exists(clean_folder):
    files = os.listdir(clean_folder)
    if files:
        with open(os.path.join(clean_folder, files[0]), 'r', encoding='utf-8') as f:
            content = f.read()
            print(f"File: {files[0]}")
            print(f"Length: {len(content)} characters")
            print(f"\nFirst 1000 characters:\n")
            print(content[:1000])
    else:
        print("No cleaned files found.")
else:
    print("Clean folder not found. Run the processor first.")

## Performance Tips

1. **Use `max_filings` for testing**: Always test with a small number first (e.g., 10-50) before downloading thousands of filings
2. **The script handles resumption**: If interrupted, already downloaded files will be skipped on restart
3. **Rate limiting**: The tool automatically adds delays between requests to respect SEC servers
4. **Download times**: Expect roughly 1-2 seconds per filing (download + cleaning)
5. **Complete populations**: A full year of 10-K filings (4,000-5,000 filings) takes 2-3 hours